# Teen Mental Health Risk Classification
## End-to-End Machine Learning Project | Google Colab & GitHub

**Objective:** Predict the dataset-defined `depression_risk` categories (`low`, `medium`, `high`) using lifestyle and wellbeing variables.

**Dataset credit:** [Teen Social Media Usage and Mental Health — sureshbeekhani (Kaggle)](https://www.kaggle.com/datasets/sureshbeekhani/teen-social-media-usage-and-mental-health)

> **Research/educational demonstration only.** The dataset labels and model outputs are not clinical diagnoses or validated mental-health risk scores. Do not use this notebook to make healthcare decisions.

**Workflow:** Data loading → data quality → EDA charts → train/test split → OneHotEncoder → SMOTE comparison → 6-model CV comparison → tuning (optional) → evaluation → save pipeline → example inference.

**Reproducibility:** Results depend on the supplied CSV and installed package versions. This notebook deliberately does not fabricate precomputed chart outputs.

## 0 — Environment setup
Run the installation cell in Google Colab. The setup attempts to use the currently available versions of scikit-learn and imbalanced-learn.

In [ ]:
%pip -q install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn joblib scipy

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.base import clone

SEED = 42
sns.set_theme(style="whitegrid")
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)
print("Environment ready.")

## 1 — Load the original CSV
Upload `Teen_Mental_Health_Dataset.csv` to Colab (or place it in your local working directory). The file is not bundled into this notebook; credit and redistribution permissions remain with the dataset owner.

In [ ]:
# Optional upload interface for Google Colab
try:
    from google.colab import files
    if not Path("Teen_Mental_Health_Dataset.csv").exists():
        files.upload()
except ImportError:
    pass

DATA_FILE = Path("Teen_Mental_Health_Dataset.csv")
if not DATA_FILE.exists():
    raise FileNotFoundError("Please upload Teen_Mental_Health_Dataset.csv before continuing.")
df = pd.read_csv(DATA_FILE)
print("Original shape:", df.shape)
display(df.head())

## 2 — Initial data inspection
Inspect feature types, missing data, duplicate records, basic statistics, and the target distribution. These checks guide cleaning and reveal the original class imbalance.

In [ ]:
df.info()
print("\nMissing values by column:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nTarget counts:\n", df["depression_risk"].value_counts())
print("\nTarget percentages:\n", (100 * df["depression_risk"].value_counts(normalize=True)).round(2))
display(df.describe(include="all"))

## 3 — Data cleaning
Remove exact duplicate rows and rows with missing data. For larger missingness, imputation should be investigated rather than automatically dropping observations. Check target classes and basic ranges. We do not claim outliers were corrected unless they actually are.

In [ ]:
df_clean = df.drop_duplicates().dropna().reset_index(drop=True).copy()
expected_labels = {"low", "medium", "high"}
actual_labels = set(df_clean["depression_risk"].unique())
if not actual_labels.issubset(expected_labels):
    raise ValueError(f"Unexpected target labels: {actual_labels - expected_labels}")
print("Before cleaning:", df.shape, "After cleaning:", df_clean.shape)
print("Remaining missing values:", df_clean.isna().sum().sum())
print("Remaining duplicates:", df_clean.duplicated().sum())

## 4 — Exploratory Data Analysis (EDA)
The following charts use the cleaned original observations, **not** synthetic SMOTE records. Charts explore associations rather than prove causation. Each figure is saved to `figures/` for a GitHub README or a presentation.

In [ ]:
risk_order = ["low", "medium", "high"]
risk_palette = {"low": "#2A9D8F", "medium": "#E9C46A", "high": "#E76F51"}
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=df_clean, x="depression_risk", order=risk_order, hue="depression_risk", palette=risk_palette, legend=False, ax=ax)
ax.set(title="Risk Category Distribution", xlabel="Dataset Risk Category", ylabel="Records")
fig.tight_layout(); fig.savefig(FIGURES / "risk_distribution.png", dpi=200); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, title in zip(axes, ["age", "sleep_hours", "daily_social_media_hours"], ["Age", "Sleep Hours", "Daily Social Media Hours"]):
    sns.histplot(df_clean[col], bins=16 if col != "age" else 7, ax=ax, color="#4682B4")
    ax.set_title(title)
fig.tight_layout(); fig.savefig(FIGURES / "demographic_lifestyle_distributions.png", dpi=200); plt.show()

In [ ]:
summary = df_clean.groupby("depression_risk")[["stress_level", "anxiety_level", "sleep_hours", "daily_social_media_hours"]].mean().reindex(risk_order)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
summary[["stress_level", "anxiety_level"]].plot(kind="bar", ax=axes[0], title="Mean Stress and Anxiety", rot=0)
summary[["sleep_hours", "daily_social_media_hours"]].plot(kind="bar", ax=axes[1], title="Mean Sleep and Social Media Hours", rot=0)
for ax in axes: ax.set_xlabel("Dataset Risk Category")
fig.tight_layout(); fig.savefig(FIGURES / "lifestyle_by_risk.png", dpi=200); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=df_clean, x="depression_risk", y="stress_level", order=risk_order, ax=axes[0], color="#78C6B8")
sns.boxplot(data=df_clean, x="depression_risk", y="sleep_hours", order=risk_order, ax=axes[1], color="#A4B8DB")
axes[0].set_title("Stress Level by Risk Category")
axes[1].set_title("Sleep Hours by Risk Category")
fig.tight_layout(); fig.savefig(FIGURES / "boxplots_by_risk.png", dpi=200); plt.show()

In [ ]:
numeric_cols_eda = df_clean.select_dtypes(include="number").columns.tolist()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_clean[numeric_cols_eda].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Heatmap — Numeric Features")
fig.tight_layout(); fig.savefig(FIGURES / "correlation_heatmap.png", dpi=200); plt.show()

## 5 — Separate features and target, then split
Use a stratified 70/30 split. The external test partition is left unchanged by encoding and resampling. The previous project used it repeatedly, so the resulting scores should be presented as **exploratory**, not a fresh independent clinical validation.

In [ ]:
X = df_clean.drop(columns=["depression_risk"])
y = df_clean["depression_risk"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Categorical:", cat_cols)
print("Numerical:", num_cols)

## 6 — One-hot encoding
Fit the encoder **only on training data**, then transform the untouched test split. The demonstration follows the historical OneHotEncoder + SMOTE workflow. A limitation: standard SMOTE may interpolate binary one-hot columns into fractional values. A future methodological improvement is to benchmark categorical-aware SMOTENC.

In [ ]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_cat = pd.DataFrame(ohe.fit_transform(X_train[cat_cols]), index=X_train.index, columns=ohe.get_feature_names_out(cat_cols))
test_cat = pd.DataFrame(ohe.transform(X_test[cat_cols]), index=X_test.index, columns=ohe.get_feature_names_out(cat_cols))
X_train_ready = pd.concat([X_train[num_cols], train_cat], axis=1)
X_test_ready = pd.concat([X_test[num_cols], test_cat], axis=1)
print("Encoded train:", X_train_ready.shape, "Encoded test:", X_test_ready.shape)
display(X_train_ready.head())

## 7 — Synthetic data augmentation: SMOTE to 10,000 training examples
Use original training records as input. The resulting 10,000 records include both real training rows and **synthetic examples**, not 10,000 independently observed people. The held-out test partition stays at its original size.

In [ ]:
smote_target = {"low": 3334, "medium": 3333, "high": 3333}
smote = SMOTE(sampling_strategy=smote_target, random_state=SEED, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_ready, y_train)
print("Training before SMOTE:", len(X_train_ready))
print("Training after SMOTE:", len(X_train_resampled))
print("Synthetic rows added:", len(X_train_resampled) - len(X_train_ready))
print("Test records unchanged:", len(X_test_ready))
print(y_train_resampled.value_counts())

## 8 — Train Random Forest without SMOTE and with SMOTE
**Controlled exploratory comparison:** identical Random Forest parameters, same one-hot features, and same test rows; the only difference is SMOTE in the training data. This test set has been consulted during previous project development, so the comparison should not be described as a definitive unbiased performance estimate.

In [ ]:
RF_PARAMS = dict(n_estimators=300, max_depth=8, min_samples_split=10,
                 min_samples_leaf=3, max_features="sqrt", random_state=SEED, n_jobs=-1)
rf_without_smote = RandomForestClassifier(**RF_PARAMS)
rf_without_smote.fit(X_train_ready, y_train)
rf_with_smote = RandomForestClassifier(**RF_PARAMS)
rf_with_smote.fit(X_train_resampled, y_train_resampled)

pred_without = rf_without_smote.predict(X_test_ready)
pred_with = rf_with_smote.predict(X_test_ready)
comparison = pd.DataFrame([
    {"Method": "Without SMOTE", "Training records": len(X_train_ready), "Test Accuracy": accuracy_score(y_test, pred_without), "Test Macro F1": f1_score(y_test, pred_without, average="macro")},
    {"Method": "With SMOTE", "Training records": len(X_train_resampled), "Test Accuracy": accuracy_score(y_test, pred_with), "Test Macro F1": f1_score(y_test, pred_with, average="macro")},
]).set_index("Method")
display(comparison.round(4))
print("Accuracy improvement (percentage points):", round((comparison.loc["With SMOTE", "Test Accuracy"] - comparison.loc["Without SMOTE", "Test Accuracy"]) * 100, 2))
print("Macro F1 improvement (percentage points):", round((comparison.loc["With SMOTE", "Test Macro F1"] - comparison.loc["Without SMOTE", "Test Macro F1"]) * 100, 2))
fig, ax = plt.subplots(figsize=(9, 5))
(comparison[["Test Accuracy", "Test Macro F1"]] * 100).plot(kind="bar", ax=ax, rot=0)
ax.set(title="Random Forest: Before vs After SMOTE", ylabel="Test score (%)", ylim=(0, 100))
fig.tight_layout(); fig.savefig(FIGURES / "before_after_smote.png", dpi=200); plt.show()

## 9 — Classification report and confusion matrix
Class-specific precision, recall, and F1 reveal mistakes hidden by a single accuracy number. Pay particular attention to the `medium` category, which was the weakest class in the previous experiments.

In [ ]:
print(classification_report(y_test, pred_with, labels=["high", "low", "medium"], zero_division=0))
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay.from_predictions(y_test, pred_with, labels=["high", "low", "medium"], cmap="Blues", values_format="d", ax=ax)
ax.set_title("Random Forest Confusion Matrix (With SMOTE)")
fig.tight_layout(); fig.savefig(FIGURES / "confusion_matrix.png", dpi=200); plt.show()

## 10 — Compare six machine learning models with 5-fold CV
Every fold independently fits its own preprocessing and SMOTE on its **training part only**. The original `X_test` is not involved. For model comparison we use **default balancing SMOTE** rather than imposing exactly 10,000 per fold. This matches the historical six-model comparison design.

**Selection metric:** mean Macro F1 across 5 validation folds. Trees are compared alongside a scaled Logistic Regression baseline.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "Random Forest": RandomForestClassifier(**RF_PARAMS),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=SEED),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
model_results = []
for name, estimator in models.items():
    prep = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("num", StandardScaler(), num_cols)
    ])
    candidate = Pipeline([
        ("preprocessing", prep),
        ("smote", SMOTE(random_state=SEED)),
        ("model", estimator)
    ])
    scores = cross_validate(candidate, X_train, y_train, cv=cv,
                            scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"}, n_jobs=1)
    model_results.append({"Model": name, "CV Accuracy": scores["test_accuracy"].mean(),
                          "CV Macro F1": scores["test_f1_macro"].mean(), "F1 Std": scores["test_f1_macro"].std()})
    print(name, "complete")
results_df = pd.DataFrame(model_results).sort_values("CV Macro F1", ascending=False).reset_index(drop=True)
display(results_df.round(4))
print("Best model in this comparison:", results_df.loc[0, "Model"])
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=results_df, x="CV Macro F1", y="Model", color="#398CB9", ax=ax)
ax.set(xlim=(0, 1), title="Model Comparison — 5-Fold CV Macro F1")
fig.tight_layout(); fig.savefig(FIGURES / "model_comparison.png", dpi=200); plt.show()

## 11 — Cross-validation of the selected baseline configuration
To compare with the 10,000-sample experiment, repeat SMOTE **inside each fold** to the same class targets. This uses five training/validation splits and prevents synthetic data from leaking across folds.

In [ ]:
preprocessing = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ("num", "passthrough", num_cols),
])
cv_pipeline = Pipeline([
    ("encoding", preprocessing),
    ("smote", SMOTE(sampling_strategy=smote_target, random_state=SEED, k_neighbors=5)),
    ("model", RandomForestClassifier(**RF_PARAMS)),
])
cv_scores = cross_validate(cv_pipeline, X_train, y_train, cv=cv,
                           scoring={"accuracy": "accuracy", "f1_macro": "f1_macro"}, n_jobs=1)
print("Mean CV accuracy:", round(cv_scores["test_accuracy"].mean(), 4))
print("Mean CV Macro F1:", round(cv_scores["test_f1_macro"].mean(), 4))
print("CV Macro F1 standard deviation:", round(cv_scores["test_f1_macro"].std(), 4))

## 12 — Optional hyperparameter tuning (slow)
The original experiment tried 20 parameter combinations × 5 folds = 100 fits. Turn the switch below on only if you want to reproduce the tuning process. The best score is selected **within the CV search**, so it can be optimistic; consider nested CV for a more rigorous estimate.

In [ ]:
RUN_TUNING = False  # Change to True to run 100 CV fits
if RUN_TUNING:
    search = RandomizedSearchCV(
        cv_pipeline,
        param_distributions={
            "model__n_estimators": [100, 200, 300, 400],
            "model__max_depth": [5, 8, 10, 12, None],
            "model__min_samples_split": [2, 5, 10, 15],
            "model__min_samples_leaf": [1, 2, 3, 5],
            "model__max_features": ["sqrt", "log2", 0.8]
        },
        n_iter=20, cv=cv, scoring="f1_macro", random_state=SEED,
        n_jobs=1, verbose=1, refit=True,
    )
    search.fit(X_train, y_train)
    print("Best parameters:", search.best_params_)
    print("Best search CV Macro F1:", round(search.best_score_, 4))
else:
    print("Tuning skipped. Set RUN_TUNING=True if desired.")

## 13 — Feature importance (baseline Random Forest)
Impurity-based importance indicates how frequently and effectively trees split on each encoded feature. This is **not** a causal interpretation or clinical evidence.

In [ ]:
importance = pd.Series(rf_with_smote.feature_importances_, index=X_train_ready.columns).sort_values(ascending=False)
display(importance.rename("Importance").to_frame())
fig, ax = plt.subplots(figsize=(10, 6))
importance.sort_values().plot(kind="barh", ax=ax, color="#489B91")
ax.set(title="Random Forest Feature Importance", xlabel="Impurity-based importance")
fig.tight_layout(); fig.savefig(FIGURES / "feature_importance.png", dpi=200); plt.show()

## 14 — Save the selected baseline model as a complete pipeline
This pipeline contains encoding + SMOTE + Random Forest. SMOTE operates during `fit()` only; it does not generate examples when predicting new rows. Only load `.pkl` files you trust.

In [ ]:
final_model = clone(cv_pipeline)
final_model.fit(X_train, y_train)
joblib.dump(final_model, "depression_risk_model.pkl")
print("Saved depression_risk_model.pkl")

## 15 — Demonstration prediction (fictional inputs)
A fictional input row is provided to test the saved model's interface. Model output probabilities are algorithmic scores, **not calibrated clinical risk probabilities**.

In [ ]:
loaded_model = joblib.load("depression_risk_model.pkl")
new_data = pd.DataFrame([{
    "age": 16, "gender": "male", "daily_social_media_hours": 5.5,
    "platform_usage": "Instagram", "sleep_hours": 6.0,
    "screen_time_before_sleep": 2.0, "academic_performance": 3.0,
    "physical_activity": 1.0, "social_interaction_level": "medium",
    "stress_level": 7, "anxiety_level": 8,
}])
print("Predicted dataset category:", loaded_model.predict(new_data)[0])
scores = pd.DataFrame({"Category": loaded_model.classes_,
                       "Model score (%)": loaded_model.predict_proba(new_data)[0] * 100})
display(scores.round(2))

## 16 — Research interpretation, limitations and next steps

- **Historical exploratory results (earlier runs):** without SMOTE, test accuracy **75.87%** and Macro F1 **72.50%**; with SMOTE, test accuracy **78.27%** and Macro F1 **76.06%**. Re-run above cells to calculate your current results from the uploaded CSV.
- **Historical six-model comparison:** Random Forest led with mean CV Macro F1 **75.67%** (comparison run used default balancing SMOTE).
- **Class imbalance:** original category counts were approximately `low 1266`, `medium 650`, `high 584`; SMOTE was applied only on training data.
- **Medium category:** the most difficult class in historical error analysis, F1 about **0.60**.
- **Limitations:** synthetic records are not real observations; ordinary SMOTE over one-hot features can create fractional indicator values; a repeatedly consulted test split is not an independent final benchmark; data provenance and labeling must be reviewed; the model has not been clinically validated.
- **Next steps:** evaluate categorical-aware SMOTENC; use an untouched independent test set or nested cross-validation; assess class-specific errors, calibration, and fairness; gather representative high-quality real data.

### References
1. [Sureshbeekhani, *Teen Social Media Usage and Mental Health*, Kaggle](https://www.kaggle.com/datasets/sureshbeekhani/teen-social-media-usage-and-mental-health)
2. [Scikit-learn documentation](https://scikit-learn.org/stable/)
3. [Imbalanced-learn documentation](https://imbalanced-learn.org/stable/)
4. [Pandas documentation](https://pandas.pydata.org/docs/)

**GitHub tip:** Upload this `.ipynb` plus `README.md`; run the notebook to generate figures under `figures/`. Do not commit private personal records, credentials, or large files unless permitted.